# EDA — Histopathologic Cancer Detection (PCam)

Sanity checks before training. Confirms the four things that most often go wrong on
this dataset:
1. **Class balance** — how skewed are the labels?
2. **Slide (WSI) structure & leakage** — why a random split inflates val AUROC.
3. **Sample tiles** — do tumor vs non-tumor patches look sane?
4. **dtype/size + cv2 decode** — patches are `96x96x3 uint8`, and cv2 gives **BGR** so
   we must convert to RGB (the pipeline does this in `src/data.py`).

Runs on Kaggle (data pre-mounted) or locally (after `scripts/download_data.sh`).

## 0. Setup — paths, imports, config

In [ ]:
import os, sys, gzip
from pathlib import Path

# Locate repo root (so `import src` + configs/ resolve whether run from root or notebooks/).
CWD = Path.cwd()
REPO = CWD if (CWD / 'configs').exists() else CWD.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import load_config
cfg = load_config(REPO / 'configs' / 'baseline.yaml')

# Data root: Kaggle mount if present, else the config's (local ./data).
ON_KAGGLE = os.path.exists('/kaggle/input')
ROOT = Path(cfg['data']['root']) if Path(cfg['data']['root']).exists() else REPO / 'data'
TRAIN_DIR = ROOT / cfg['data']['train_dir']
LABELS = ROOT / cfg['data']['labels_csv']
EXT = cfg['data']['image_ext']
print('Kaggle' if ON_KAGGLE else 'local/Colab', '| ROOT =', ROOT)
print('labels exists:', LABELS.exists(), '| train dir exists:', TRAIN_DIR.exists())

## 1. Class balance
Label = 1 if the center 32x32 region of the 96x96 patch contains >=1 tumor pixel.

In [ ]:
labels = pd.read_csv(LABELS)
counts = labels['label'].value_counts().sort_index()
pos_rate = labels['label'].mean()
print(f'patches: {len(labels):,}')
print(counts.to_string())
print(f'positive (tumor) rate: {pos_rate:.3f}')

ax = counts.rename({0: 'no-tumor (0)', 1: 'tumor (1)'}).plot.bar(rot=0, color=['#4c72b0', '#c44e52'])
ax.set_title(f'Class balance — pos rate {pos_rate:.1%}'); ax.set_ylabel('patches')
plt.tight_layout(); plt.show()

## 2. Slide (WSI) structure & leakage
Patches are cropped from a small set of whole-slide images. If patches from one slide
land in **both** train and val, near-duplicates leak and val AUROC is optimistic. We
split by slide (`GroupShuffleSplit`) to prevent it — see the bundled map.

In [ ]:
wsi_path = REPO / 'data' / 'wsi' / 'patch_id_wsi_full.csv.gz'
if wsi_path.exists():
    wsi = pd.read_csv(wsi_path)  # id,wsi (reads .gz transparently)
    per_slide = wsi.groupby('wsi').size()
    print(f'patches: {len(wsi):,} across {wsi["wsi"].nunique()} slides')
    print(f'patches/slide: min={per_slide.min()} max={per_slide.max()} median={per_slide.median():.0f}')
    kind = wsi['wsi'].str.extract(r'(normal|tumor)')[0].value_counts()
    print('slide types:', dict(kind))

    m = labels.merge(wsi, on='id', how='left')
    print('label ids covered by map:', int(m['wsi'].notna().sum()), '/', len(labels))
    slide_posrate = m.groupby('wsi')['label'].mean()

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    per_slide.plot.hist(bins=40, ax=ax[0], color='#4c72b0')
    ax[0].set_title('patches per slide'); ax[0].set_xlabel('patches')
    slide_posrate.plot.hist(bins=30, ax=ax[1], color='#c44e52')
    ax[1].set_title('per-slide tumor rate'); ax[1].set_xlabel('fraction tumor')
    plt.tight_layout(); plt.show()
else:
    print('WSI map not found at', wsi_path, '- grouped split will fall back to stratified (a warning is logged).')

## 3. Sample tiles — tumor vs non-tumor

In [ ]:
import cv2  # opencv-python-headless

def load_rgb(patch_id):
    img = cv2.imread(str(TRAIN_DIR / f'{patch_id}{EXT}'), cv2.IMREAD_COLOR)  # BGR uint8
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

if TRAIN_DIR.exists():
    n = 6
    fig, axes = plt.subplots(2, n, figsize=(2*n, 4))
    for row, lab in enumerate([0, 1]):
        ids = labels[labels['label'] == lab]['id'].head(n).tolist()
        for col, pid in enumerate(ids):
            img = load_rgb(pid)
            axes[row, col].imshow(img); axes[row, col].axis('off')
        axes[row, 0].set_ylabel(['no-tumor', 'tumor'][lab], rotation=0, labelpad=30, va='center')
    fig.suptitle('Top row: non-tumor   Bottom row: tumor'); plt.tight_layout(); plt.show()
else:
    print('Train images not available here (SandboxPi has no image data) - run this cell on Kaggle/Colab.')

## 4. dtype / size check + cv2 decode verification
Asserts decoded patches are `96x96x3 uint8`, and shows why the **BGR->RGB** conversion
in `src/data.py` matters (raw cv2 output is BGR; feeding it as RGB swaps red/blue).

In [ ]:
if TRAIN_DIR.exists():
    sample_ids = labels['id'].head(200).tolist()
    bad = []
    for pid in sample_ids:
        raw = cv2.imread(str(TRAIN_DIR / f'{pid}{EXT}'), cv2.IMREAD_COLOR)
        assert raw is not None, f'unreadable: {pid}'
        if raw.shape != (96, 96, 3) or raw.dtype != np.uint8:
            bad.append((pid, raw.shape, str(raw.dtype)))
    print(f'checked {len(sample_ids)} patches; off-spec (not 96x96x3 uint8): {len(bad)}')
    assert not bad, bad[:5]

    pid = sample_ids[0]
    bgr = cv2.imread(str(TRAIN_DIR / f'{pid}{EXT}'), cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    ax[0].imshow(bgr); ax[0].set_title('raw cv2 (BGR) - WRONG'); ax[0].axis('off')
    ax[1].imshow(rgb); ax[1].set_title('BGR->RGB - correct'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
    print('channel means  BGR:', bgr.reshape(-1,3).mean(0).round(1),
          ' RGB:', rgb.reshape(-1,3).mean(0).round(1))
    print('dtype:', rgb.dtype, '| shape:', rgb.shape, '| range:', rgb.min(), '-', rgb.max(), '(raw [0,255]; NO manual rescale)')
else:
    print('Train images not available here - run on Kaggle/Colab to verify decode.')

## Takeaways
- Class balance is moderate (~40% positive) — AUROC is the right metric; no resampling needed for a baseline.
- Patches come from a **small number of slides** → always use the **WSI-grouped split** (bundled map, on by default).
- Decoded patches are `96x96x3 uint8` in `[0,255]`; `include_preprocessing=True` handles normalization —
  **do not** add any manual rescaling, or you double-normalize and tank AUROC.